# CSTS: Listen to Look into the Future
## Audio-Visual Egocentric Gaze Anticipation — Full Tutorial

**Paper**: *Listen to Look into the Future: Audio-Visual Egocentric Gaze Anticipation* (ECCV 2024)

---

### What does this model do?

> Given a short egocentric video clip + synchronized audio, predict **where the person will look next** — output is a spatial heatmap over future frames.

```
        Video frames (RGB)    Audio (STFT spectrogram)
              ↓                        ↓
       [Vision Encoder]          [Audio Encoder]
              ↓                        ↓
         Spatial Fusion  ←→   Temporal Fusion
                       ↓
              [Transformer Decoder]
                       ↓
         Gaze Heatmap  (B, 1, T, H, W)
```

### Tutorial Outline
1. Environment Setup
2. Data Format & Input Shapes
3. Model Architecture (CSTS)
4. Loss Functions (KLDiv + EgoNCE)
5. Optimizer & Learning Rate Schedule
6. Training Loop
7. Evaluation & Metrics
8. Running Inference
9. Config System
10. Putting It All Together

---
## Section 1 — Environment Setup

In [ ]:
# ── Install dependencies (run once in Colab) ──────────────────────────────────
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install fvcore iopath pytorchvideo einops timm
# !pip install av  # PyAV for video decoding

# ── Clone the repository ──────────────────────────────────────────────────────
# !git clone https://github.com/<repo>/CSTS-main.git
# %cd CSTS-main

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device    : {DEVICE}')

---
## Section 2 — Data Format & Input Shapes

### 2.1 What the dataset provides

The dataset classes (`Ego4d_av_gaze_forecast`, `Aria_av_gaze_forecast`) return a 6-tuple per sample:

| Field | Shape | Description |
|-------|-------|-------------|
| `frames` | `(3, T, H, W)` | RGB video, T=8 frames, H=W=256 |
| `audio_frames` | `(1, F, L)` | STFT spectrogram, F=128 freq bins, L=256 time steps |
| `labels` | `(T, 3)` | Per-frame gaze `[x, y, gaze_type]` (normalized 0–1) |
| `labels_hm` | `(T, H/4, W/4)` | Gaussian heatmap targets at 64×64 |
| `index` | scalar | Video clip index |
| `meta` | dict | Paths & frame indices |

After collation into a batch of size B:

| Tensor | Batch Shape |
|--------|-------------|
| `frames` | `(B, 3, 8, 256, 256)` |
| `audio_frames` | `(B, 1, 128, 256)` |
| `labels` | `(B, 8, 3)` |
| `labels_hm` | `(B, 8, 64, 64)` |

In [ ]:
# ── 2.2  Simulate a batch (no dataset needed) ─────────────────────────────────
B  = 2    # batch size
T  = 8    # temporal frames
H  = W  = 256   # spatial resolution
F_bins = 128    # STFT frequency bins
L_time = 256    # STFT time steps

frames       = torch.randn(B, 3,      T, H, W)         # RGB video
audio_frames = torch.randn(B, 1, F_bins, L_time)        # audio spectrogram
labels_hm    = torch.rand (B, T, H//4, W//4)            # heatmap targets
# Normalize heatmaps so each frame sums to 1 (probability map)
labels_hm    = labels_hm / (labels_hm.sum(dim=(-2,-1), keepdim=True) + 1e-8)

print('frames       :', frames.shape)        # (2, 3, 8, 256, 256)
print('audio_frames :', audio_frames.shape)  # (2, 1, 128, 256)
print('labels_hm    :', labels_hm.shape)     # (2, 8, 64, 64)

In [ ]:
# ── 2.3  Gaussian heatmap generation (from the dataset class) ─────────────────
import cv2

def make_gaussian_heatmap(x_norm, y_norm, H=64, W=64, kernel_size=19):
    """Create a 2-D Gaussian heatmap for a single fixation.

    Args:
        x_norm, y_norm : gaze coordinates in [0, 1]
        H, W           : output heatmap size (pixels)
        kernel_size    : size of the Gaussian kernel (must be odd)
    Returns:
        heatmap : (H, W) numpy array, sums to 1
    """
    heatmap = np.zeros((H, W), dtype=np.float32)
    cx = int(x_norm * W)
    cy = int(y_norm * H)
    # Clamp to valid range
    cx = max(0, min(W - 1, cx))
    cy = max(0, min(H - 1, cy))
    heatmap[cy, cx] = 1.0
    heatmap = cv2.GaussianBlur(heatmap, (kernel_size, kernel_size), 0)
    total = heatmap.sum()
    if total > 0:
        heatmap /= total
    return heatmap

# Example: gaze at center of frame
hm = make_gaussian_heatmap(0.5, 0.5)
print('Heatmap shape :', hm.shape)   # (64, 64)
print('Sums to       :', hm.sum())   # ~1.0

---
## Section 3 — Model Architecture

### 3.1 Overview

```
                     ┌─────────────────────────────────────────────────┐
  frames             │           CSTS Model                            │
(B,3,8,256,256) ─→  │  PatchEmbed  →  16 Vision Blocks                │
                     │                      ↓                          │
audio_frames   ─→   │  PatchEmbed  →  4 Audio Blocks                  │
(B,1,128,256)        │                      ↓                          │
                     │      Spatial Fusion (concat+attn)               │
                     │      Temporal Fusion (concat+attn+reweight)     │
                     │                      ↓                          │
                     │      4 Decoder Blocks (upsampling)              │
                     │                      ↓                          │
                     │      Conv3d(96→1)  →  (B,1,8,64,64)            │
                     └─────────────────────────────────────────────────┘
```

### 3.2 Patch Embedding

Both video frames and audio spectrograms are tokenized with a 3-D convolution (treating time as a third spatial axis).

| Parameter | Value |
|-----------|-------|
| Kernel | `(3, 7, 7)` |
| Stride | `(2, 4, 4)` |
| Padding | `(1, 3, 3)` |
| Output dim | 96 |

For an input `(B, C, 8, 256, 256)` → output `(B, 4×64×64, 96) = (B, 16384, 96)`

In [ ]:
# ── 3.3  PatchEmbed (simplified stand-alone version) ──────────────────────────
class PatchEmbed(nn.Module):
    """3-D patch embedding used for both vision and audio streams.

    Reference: slowfast/models/custom_multimodal_builder.py  (lines 66-88)
    """
    def __init__(self, in_channels=3, embed_dim=96,
                 kernel=(3,7,7), stride=(2,4,4), padding=(1,3,3)):
        super().__init__()
        self.proj = nn.Conv3d(
            in_channels, embed_dim,
            kernel_size=kernel, stride=stride, padding=padding
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x : (B, C, T, H, W)
        x = self.proj(x)                         # (B, embed_dim, T', H', W')
        B, C, T, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)         # (B, T'*H'*W', embed_dim)
        x = self.norm(x)
        return x, (T, H, W)                      # tokens + spatial dims

# Vision patch embed (3-channel RGB)
vis_embed = PatchEmbed(in_channels=3, embed_dim=96)
out_v, dims_v = vis_embed(frames)                # frames: (B,3,8,256,256)
print(f'Vision tokens : {out_v.shape}')          # (2, 16384, 96)
print(f'T,H,W after embed : {dims_v}')           # (4, 64, 64)

# Audio patch embed (1-channel spectrogram)
aud_embed = PatchEmbed(in_channels=1, embed_dim=96)
# Expand audio to match shape (B,1,T,H,W) — audio is treated as a video
audio_3d = audio_frames.expand(-1, -1, T, -1, -1)  # (B,1,8,128,256) → use as-is
# Actual shape used: (B, 1, 8, 256, 256) in the paper — pad or resize audio to 256×256
audio_padded = F.interpolate(audio_frames.unsqueeze(2).expand(-1,-1,T,-1,-1).float(),
                              size=(T,256,256), mode='trilinear', align_corners=False)
out_a, dims_a = aud_embed(audio_padded)          # (B,1,8,256,256)
print(f'Audio tokens  : {out_a.shape}')          # (2, 16384, 96)

### 3.4 Positional Embeddings

CSTS uses **separate** spatial and temporal positional embeddings (not combined sinusoidal).

```
token_pos = pos_embed_spatial[spatial_idx] + pos_embed_temporal[temporal_idx]
```

| Embedding | Shape |
|-----------|-------|
| `pos_embed_spatial` | `(1, 64×64, 96)` = `(1, 4096, 96)` |
| `pos_embed_temporal` | `(1, 4, 96)` |
| `pos_embed_spatial_audio` | `(1, 4096, 96)` |
| `pos_embed_temporal_audio` | `(1, 4, 96)` |

In [ ]:
# ── 3.5  Separate positional embeddings ───────────────────────────────────────
# Reference: slowfast/models/custom_multimodal_builder.py  (lines 92-112)

embed_dim = 96
# After patch embed: T'=4, H'=64, W'=64, so spatial = 64*64 = 4096, temporal = 4
T_p, H_p, W_p = 4, 64, 64

pos_embed_spatial   = nn.Parameter(torch.zeros(1, H_p * W_p, embed_dim))  # (1, 4096, 96)
pos_embed_temporal  = nn.Parameter(torch.zeros(1, T_p,        embed_dim))  # (1, 4, 96)
nn.init.trunc_normal_(pos_embed_spatial,  std=0.02)
nn.init.trunc_normal_(pos_embed_temporal, std=0.02)

def add_pos_embed(tokens, pos_spatial, pos_temporal, T, H, W):
    """Add factorized spatial+temporal positional embeddings.

    Args:
        tokens      : (B, T*H*W, C)
        pos_spatial : (1, H*W, C)
        pos_temporal: (1, T, C)
        T, H, W     : patch-grid dimensions
    """
    B, N, C = tokens.shape
    # Spatial embed: repeat T times across temporal axis
    # (1, H*W, C) → (1, T*H*W, C)
    sp = pos_spatial.unsqueeze(1).expand(-1, T, -1, -1)  # (1, T, H*W, C)
    sp = sp.reshape(1, T * H * W, C)                     # (1, T*H*W, C)
    # Temporal embed: repeat H*W times across spatial axis
    # (1, T, C) → (1, T, H*W, C) → (1, T*H*W, C)
    tp = pos_temporal.unsqueeze(2).expand(-1, -1, H * W, -1).reshape(1, T * H * W, C)
    return tokens + sp + tp

tokens_v_pos = add_pos_embed(out_v, pos_embed_spatial, pos_embed_temporal, T_p, H_p, W_p)
print('Tokens with pos embed:', tokens_v_pos.shape)  # (2, 16384, 96)

### 3.5 Transformer Encoder Blocks

CSTS uses **MViT-style MultiScaleBlocks** — attention with Q/K/V pooling to progressively reduce resolution while expanding channels.

#### Vision Encoder (16 blocks)

| Stage | Block idx | Embed dim | Num heads | Q-stride |
|-------|-----------|-----------|-----------|----------|
| Block1 | 0 | 96 | 1 | `[1,1,1,1]` (no pool) |
| Block2 | 1–3 | 96→192 | 1→2 | `[1,1,2,2]` at idx=1,3 |
| Block3 | 3–14 | 192→384 | 2→4 | `[1,1,2,2]` at idx=3 |
| Block4 | 14–16 | 384→768 | 4→8 | `[1,1,1,1]` |

KV pooling: adaptive stride `[1, 8, 8]` (pools K and V spatially)

#### Audio Encoder (4 blocks)

| Block | Input dim | Output dim | Heads | Q-stride |
|-------|-----------|------------|-------|----------|
| 0 | 96 | 192 | 1 | `[]` |
| 1 | 192 | 384 | 2 | `[1,2,2]` |
| 2 | 384 | 768 | 4 | `[1,2,2]` |
| 3 | 768 | 768 | 8 | `[1,2,2]` |

In [ ]:
# ── 3.6  Simplified MultiScaleBlock (educational stand-alone) ─────────────────
# Full version: slowfast/models/custom_multimodal_builder.py
# This stripped-down version illustrates the core idea.

class SimplifiedMultiScaleBlock(nn.Module):
    """Illustrates Q/KV pooling in a MultiScale Vision Transformer block.
    This is a teaching simplification — the real block is in the codebase.
    """
    def __init__(self, dim_in, dim_out, num_heads,
                 q_pool_kernel=(1,2,2), mlp_ratio=4.0):
        super().__init__()
        self.norm1   = nn.LayerNorm(dim_in)
        self.attn    = nn.MultiheadAttention(dim_in, num_heads, batch_first=True)
        self.norm2   = nn.LayerNorm(dim_in)
        mlp_dim      = int(dim_in * mlp_ratio)
        self.mlp     = nn.Sequential(
            nn.Linear(dim_in, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, dim_in)
        )
        # Optional dim projection when dim_in != dim_out
        self.proj = nn.Linear(dim_in, dim_out) if dim_in != dim_out else nn.Identity()
        # Q pooling: compress query sequence (reduces token count)
        if any(k > 1 for k in q_pool_kernel):
            self.q_pool = nn.AvgPool3d(q_pool_kernel, stride=q_pool_kernel, padding=0)
        else:
            self.q_pool = None

    def forward(self, x):
        # x : (B, N, C)
        h = self.norm1(x)
        # In real MViT, Q is spatially pooled but K/V are kept or pooled differently.
        # Here we simplify: standard self-attention then optional projection.
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        x = self.proj(x)
        return x

# Quick forward check
blk = SimplifiedMultiScaleBlock(dim_in=96, dim_out=192, num_heads=1)
dummy = torch.randn(2, 16384, 96)  # (B, N, C)
out   = blk(dummy)
print('After SimplifiedMultiScaleBlock:', out.shape)  # (2, 16384, 192)

### 3.6 Audio–Visual Fusion

After the encoders the model fuses modalities in two complementary ways:

#### Spatial Fusion
1. Audio tokens are **spatially pooled** to a small set of anchor tokens.
2. Audio anchors are **concatenated** with vision tokens.
3. A shared `SpatialBlock` (multi-head attention) lets vision tokens attend to audio.
4. Vision and audio parts are split back.

#### Temporal Fusion
1. Both modalities are **temporally pooled** to per-frame summaries.
2. Summaries are **concatenated** and processed by a `TemporalBlock`.
3. Temporal attention weights are used to **reweight** spatial vision features.

In [ ]:
# ── 3.7  Fusion sketch ────────────────────────────────────────────────────────
# Reference: slowfast/models/custom_multimodal_builder.py  (lines 414-461)

# After encoders (both streams at 768-d, N=256 tokens representing 4×8×8 or similar)
N_vis = 256   # vision tokens after all encoder stages
N_aud = 256   # audio tokens after all encoder stages
C_enc = 768

vis_tokens = torch.randn(B, N_vis, C_enc)  # (2, 256, 768)
aud_tokens = torch.randn(B, N_aud, C_enc)  # (2, 256, 768)

# ─── Spatial Fusion ───────────────────────────────────────────────────────────
# 1. Pool audio to fewer anchor tokens (Conv3d kernel=(1,8,8) in the real model)
N_aud_pooled = 4  # after pooling
aud_anchors  = aud_tokens[:, :N_aud_pooled, :]  # simplified: take first 4 tokens

# 2. Concatenate vision + audio anchors
fused_spatial = torch.cat([vis_tokens, aud_anchors], dim=1)  # (2, 260, 768)
print('Spatial concat:', fused_spatial.shape)

# 3. Attention (SpatialBlock)
spatial_attn = nn.MultiheadAttention(C_enc, num_heads=8, batch_first=True)
fused_spatial, _ = spatial_attn(fused_spatial, fused_spatial, fused_spatial)
print('After spatial attn:', fused_spatial.shape)  # (2, 260, 768)

# 4. Split back
vis_out_sp  = fused_spatial[:, :N_vis, :]
aud_out_sp  = fused_spatial[:, N_vis:, :]
print('Vision out (spatial):', vis_out_sp.shape)  # (2, 256, 768)

# ─── Temporal Fusion ──────────────────────────────────────────────────────────
# 1. Pool each modality to T=4 temporal tokens
vis_temp = vis_tokens.mean(dim=1, keepdim=True).expand(-1, 4, -1)   # (2, 4, 768) simplified
aud_temp = aud_tokens.mean(dim=1, keepdim=True).expand(-1, 4, -1)   # (2, 4, 768)

# 2. Concatenate
fused_temporal = torch.cat([vis_temp, aud_temp], dim=1)  # (2, 8, 768)
print('Temporal concat:', fused_temporal.shape)

# 3. Attention (TemporalBlock)
temporal_attn = nn.MultiheadAttention(C_enc, num_heads=8, batch_first=True)
fused_temporal, _ = temporal_attn(fused_temporal, fused_temporal, fused_temporal)
print('After temporal attn:', fused_temporal.shape)  # (2, 8, 768)

# 4. Use first T=4 tokens as weights to reweight vision spatial features
temp_weights = fused_temporal[:, :4, :]          # (2, 4, 768)
temp_weights = temp_weights.unsqueeze(2)         # (2, 4, 1, 768) → for broadcasting over H*W
vis_reweighted = vis_out_sp.unsqueeze(1) * temp_weights  # broadcast over spatial dim
print('Vision reweighted:', vis_reweighted.shape)

### 3.7 Transformer Decoder (4 Upsampling Blocks)

The decoder progressively **upsamples** the fused representation back to spatial resolution for heatmap prediction.

| Block | Input dim | Output dim | Heads | Q-stride (upsample) |
|-------|-----------|------------|-------|--------------------|
| Dec 1 | 768 | 768 | 8 | `[1,2,2]` |
| Dec 2 | 768 | 384 | 4 | `[1,2,2]` |
| Dec 3 | 384 | 192 | 4 | `[1,2,2]` |
| Dec 4 | 192 | 96  | 2 | `[2,1,1]` |

Final token shape before Conv3d head: `(B, 32768, 96)` → reshaped to `(B, 96, 8, 64, 64)`

In [ ]:
# ── 3.8  Decoder + Classifier head ────────────────────────────────────────────
# Reference: slowfast/models/custom_multimodal_builder.py  (lines 270-301, 462-498)

class GazeDecoder(nn.Module):
    """Simplified decoder: 4 linear projection stages with upsampling + final Conv3d."""
    def __init__(self):
        super().__init__()
        # Each stage doubles spatial resolution: project dims down, upsample tokens
        self.stage1 = nn.Linear(768, 768)
        self.stage2 = nn.Linear(768, 384)
        self.stage3 = nn.Linear(384, 192)
        self.stage4 = nn.Linear(192,  96)
        self.norm   = nn.LayerNorm(96)
        # Final classifier: map 96 channels → 1 channel (heatmap logit)
        self.classifier = nn.Conv3d(96, 1, kernel_size=1)

    def forward(self, x, T=8, H=64, W=64):
        """
        x : (B, N, C)  — fused tokens from encoder+fusion
        Returns: (B, 1, T, H, W)  — gaze heatmap logits
        """
        # In the real model, each stage includes Multi-Scale attention with Q-pooling.
        # Here we do linear projections only to illustrate dimension flow.
        x = F.gelu(self.stage1(x))     # (B, N, 768)
        x = F.gelu(self.stage2(x))     # (B, N, 384)
        x = F.gelu(self.stage3(x))     # (B, N, 192)
        x = F.gelu(self.stage4(x))     # (B, N, 96)
        x = self.norm(x)
        # Reshape token sequence → 5-D feature map
        B, N, C = x.shape
        x = x.transpose(1, 2).reshape(B, C, T, H, W)  # (B, 96, T, H, W)
        x = self.classifier(x)                         # (B, 1, T, H, W)
        return x

decoder = GazeDecoder()
# Feed 256 fused tokens (B, 256, 768)
fused = torch.randn(B, 256, 768)
# The real decoder expands tokens via Q-upsampling to 8×64×64=32768 before the head.
# For this demo, we use a smaller spatial grid matching the token count:
# 256 = 4×8×8; let's pretend T=4, H=8, W=8 for the decode demo
out_dec = decoder(fused, T=4, H=8, W=8)
print('Decoder output:', out_dec.shape)  # (2, 1, 4, 8, 8)

### 3.8 Final Model Output Shape

After a full forward pass through the real CSTS model:

```
Input:   frames (B, 3, 8, 256, 256)
         audio  (B, 1, 8, 256, 256)   [audio resized to match video spatial dims]
           ↓
Output:  preds  (B, 1, 8,  64,  64)   [logits before softmax]
```

During evaluation, a **per-frame softmax** is applied with temperature τ=2:
```python
preds_flat = preds.view(B, T, -1) / temperature          # (B, T, H*W)
preds_prob = F.softmax(preds_flat, dim=-1).view(B,1,T,H,W)
```

---
## Section 4 — Loss Functions

### 4.1 KL Divergence Loss (primary)

The model is trained to produce a spatial probability distribution matching the ground-truth Gaussian heatmap.

$$\mathcal{L}_{KL} = \frac{1}{T \cdot \log(H \cdot W)} \sum_{t=1}^{T} KL\bigl(\hat{p}_t \;\|\; p_t\bigr)$$

where $\hat{p}_t$ is the predicted probability map for frame $t$ and $p_t$ is the Gaussian heatmap target.

### 4.2 EgoNCE (contrastive, auxiliary)

Encourages cross-modal alignment between visual and audio embeddings within a batch:

$$\mathcal{L}_{NCE} = -\frac{1}{B}\sum_{i=1}^{B} \log \frac{\exp(\mathbf{v}_i \cdot \mathbf{a}_i / \tau)}{\sum_{j=1}^{B} \exp(\mathbf{v}_i \cdot \mathbf{a}_j / \tau)}$$

### 4.3 Combined Loss

$$\mathcal{L} = \mathcal{L}_{KL} + \alpha \cdot \mathcal{L}_{NCE}$$

Default: **α = 0.05** for Ego4D, **α = 0.01** for Ego4D estimation.

**Reference**: `slowfast/models/losses.py` lines 51–82, 152–170

In [ ]:
# ── 4.4  Loss implementations (matching the codebase) ─────────────────────────
# Reference: slowfast/models/losses.py

class KLDiv(nn.Module):
    """KL Divergence loss for gaze heatmap prediction.

    Reference: slowfast/models/losses.py  (lines 51-82)

    Args:
        pred   : (B, 1, T, H, W) — raw logits from model
        target : (B,    T, H, W) — normalized Gaussian heatmap (sums to 1 per frame)
    """
    def forward(self, pred, target):
        B, _, T, H, W = pred.shape
        # Flatten spatial dims and compute per-frame softmax
        pred_flat = pred.view(B, T, -1)           # (B, T, H*W)
        pred_prob = F.softmax(pred_flat, dim=-1)  # probability distribution
        target_flat = target.view(B, T, -1)       # (B, T, H*W)
        # KL divergence: sum over spatial, mean over batch & time
        eps = 1e-8
        kl = (target_flat * (torch.log(target_flat + eps) - torch.log(pred_prob + eps)))
        kl = kl.sum(dim=-1)    # (B, T)
        # Normalise by T * log(H*W) to be scale-invariant
        kl = kl / (T * math.log(H * W))
        return kl.mean()


class EgoNCE(nn.Module):
    """Contrastive loss on cross-modal similarity matrix.

    Reference: slowfast/models/losses.py  (lines 152-170)

    Args:
        x : (B, B) — similarity matrix (v_embed @ a_embed.T)
    """
    def __init__(self, temperature=0.05):
        super().__init__()
        self.tau = temperature

    def forward(self, x):
        # x : (B, B) — rows = vision, cols = audio
        x = x / self.tau
        labels = torch.arange(x.size(0), device=x.device)  # diagonal = positives
        loss = F.cross_entropy(x, labels)
        return loss


# ── 4.5  Demonstrate combined loss ────────────────────────────────────────────
kldiv_loss = KLDiv()
egonce_loss = EgoNCE(temperature=0.05)
alpha = 0.05   # MODEL.LOSS_ALPHA

# Fake predictions and targets
preds_logits = torch.randn(B, 1, T, 64, 64)  # model output
targets_hm   = torch.rand (B,    T, 64, 64)  # ground-truth heatmaps
targets_hm   = targets_hm / (targets_hm.sum(dim=(-2,-1), keepdim=True) + 1e-8)

# KL divergence part
loss_kl = kldiv_loss(preds_logits, targets_hm)
print(f'KLDiv loss  : {loss_kl.item():.4f}')

# EgoNCE part (needs cross-modal embeddings)
v_embed = F.normalize(torch.randn(B, 256), dim=-1)  # vision embedding
a_embed = F.normalize(torch.randn(B, 256), dim=-1)  # audio  embedding
sim_matrix = v_embed @ a_embed.T                     # (B, B)
loss_nce = egonce_loss(sim_matrix)
print(f'EgoNCE loss : {loss_nce.item():.4f}')

# Combined
total_loss = loss_kl + alpha * loss_nce
print(f'Total loss  : {total_loss.item():.4f}')

---
## Section 5 — Optimizer & Learning Rate Schedule

### 5.1 AdamW with parameter groups

| Parameter group | Weight decay |
|----------------|--------------|
| BN parameters | 0.0 |
| 1-D params (bias, norm) | 0.0 |
| Positional embeddings | 0.0 |
| All other params | 0.05 |

### 5.2 Cosine Annealing (no warmup)

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{\pi t}{T_{\max}}\right)$$

| Config | Value |
|--------|-------|
| BASE_LR | 1e-4 |
| COSINE_END_LR | 1e-6 |
| MAX_EPOCH | 15 |
| WARMUP_EPOCHS | 0 |

**Reference**: `slowfast/utils/lr_policy.py` lines 29–52, `slowfast/models/optimizer.py` lines 11–108

In [ ]:
# ── 5.3  Optimizer + LR schedule demo ─────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')  # Colab-safe backend
import matplotlib.pyplot as plt

# ── Parameter grouping ────────────────────────────────────────────────────────
def build_optimizer(model, base_lr=1e-4, weight_decay=0.05):
    """Build AdamW optimizer with per-group weight decay.
    Reference: slowfast/models/optimizer.py  (lines 11-108)
    """
    decay_params    = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # 1-D tensors, bias terms, and pos-embed → no weight decay
        if param.ndim == 1 or 'pos_embed' in name or 'bias' in name:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    param_groups = [
        {'params': decay_params,    'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(param_groups, lr=base_lr, betas=(0.9, 0.999), eps=1e-8)

# ── Cosine LR schedule ────────────────────────────────────────────────────────
def cosine_lr(cur_epoch, max_epoch=15, base_lr=1e-4, end_lr=1e-6):
    """Cosine annealing without warmup.
    Reference: slowfast/utils/lr_policy.py  (lines 29-52)
    """
    return end_lr + (base_lr - end_lr) * (1 + math.cos(math.pi * cur_epoch / max_epoch)) / 2

# Plot LR curve
epochs = list(range(16))
lrs    = [cosine_lr(e) for e in epochs]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(epochs, lrs, marker='o', color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule (15 epochs, 1e-4 → 1e-6)')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lr_schedule.png', dpi=120)
plt.show()
print('LR at epoch 0 :', lrs[0])
print('LR at epoch 7 :', round(lrs[7], 8))
print('LR at epoch 15:', lrs[15])

---
## Section 6 — Training Loop

### 6.1 Outer training loop (`train` function)

```
tools/train_avgaze_net.py  lines 246–361

for epoch in range(start_epoch, MAX_EPOCH):
    shuffle_dataset(train_loader, epoch)
    train_epoch(train_loader, model, optimizer, scaler, ...)
    if is_checkpoint_epoch(epoch):
        save_checkpoint(...)
    if is_eval_epoch(epoch):
        eval_epoch(val_loader, model, val_meter, ...)
```

### 6.2 Inner training step (per iteration)

```
1. Load batch: (frames, audio, labels, labels_hm, idx, meta)
2. Update LR for current step
3. Forward pass (mixed precision AMP)
   ├─ model(frames, audio, return_embed=True)
   │   └─ returns [preds, v_embed, a_embed]
   ├─ frame-wise softmax (temperature=2)
   ├─ similarity matrix = v_embed @ a_embed.T
   └─ loss = KLDiv(preds, hm) + α * EgoNCE(sim)
4. Backward + gradient clip (L2 norm ≤ 1.0)
5. optimizer.step() + scaler.update()
6. Metrics: adaptive_f1(preds, labels_hm, labels)
7. Log to TensorBoard
```

In [ ]:
# ── 6.3  Training loop skeleton (mirrors tools/train_avgaze_net.py) ───────────

def frame_softmax(preds, temperature=2.0):
    """Apply per-frame softmax to heatmap logits.

    Args:
        preds : (B, 1, T, H, W) raw logits
    Returns:
        (B, 1, T, H, W) probability maps
    """
    B, _, T, H, W = preds.shape
    preds_flat = preds.view(B, T, -1) / temperature  # (B, T, H*W)
    preds_prob = F.softmax(preds_flat, dim=-1)        # per-frame probs
    return preds_prob.view(B, 1, T, H, W)


def train_one_epoch(model, loader_mock, optimizer, scaler,
                    kldiv_fn, egonce_fn, alpha=0.05, clip_norm=1.0,
                    device='cpu'):
    """One training epoch.
    Reference: tools/train_avgaze_net.py  train_epoch()  (lines 25-156)
    """
    model.train()
    total_loss = 0.0

    for batch_idx, batch in enumerate(loader_mock):
        frames, audio, labels, labels_hm = [x.to(device) for x in batch]

        # ── Forward (mixed precision) ─────────────────────────────────────────
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            # In the real model: preds, v_embed, a_embed = model(frames, audio, return_embed=True)
            # Here we simulate:
            preds   = torch.randn(frames.size(0), 1, T, 64, 64, device=device)
            v_embed = F.normalize(torch.randn(frames.size(0), 256, device=device), dim=-1)
            a_embed = F.normalize(torch.randn(frames.size(0), 256, device=device), dim=-1)

            # Temperature-scaled softmax → probability map
            preds_prob = frame_softmax(preds, temperature=2.0)

            # Losses
            loss_kl  = kldiv_fn(preds_prob, labels_hm)
            sim      = v_embed @ a_embed.T            # (B, B) similarity matrix
            loss_nce = egonce_fn(sim)
            loss     = loss_kl + alpha * loss_nce

        # ── Backward ─────────────────────────────────────────────────────────
        optimizer.zero_grad()
        if device == 'cuda':
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()

        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx:3d} | loss={loss.item():.4f} '
                  f'(kl={loss_kl.item():.4f}, nce={loss_nce.item():.4f})')

    return total_loss / len(loader_mock)


# ── Mock dataloader ────────────────────────────────────────────────────────────
def make_mock_loader(n_batches=5, batch_size=2):
    """Yield (frames, audio, labels, labels_hm) batches."""
    for _ in range(n_batches):
        frames   = torch.randn(batch_size, 3, T, H, W)
        audio    = torch.randn(batch_size, 1, F_bins, L_time)
        labels   = torch.rand (batch_size, T, 3)
        hm       = torch.rand (batch_size, T, 64, 64)
        hm       = hm / (hm.sum(dim=(-2,-1), keepdim=True) + 1e-8)
        yield frames, audio, labels, hm


# ── Run 1 epoch demo ──────────────────────────────────────────────────────────
demo_model = nn.Linear(10, 10)   # placeholder model for optimizer demo
demo_opt   = torch.optim.AdamW(demo_model.parameters(), lr=1e-4, weight_decay=0.05)
demo_scaler = torch.cuda.amp.GradScaler(enabled=False)  # disabled on CPU

kldiv_fn  = KLDiv()
egonce_fn = EgoNCE(temperature=0.05)

print('=== Training epoch demo ===')
avg_loss = train_one_epoch(
    model=demo_model,
    loader_mock=make_mock_loader(n_batches=3, batch_size=2),
    optimizer=demo_opt,
    scaler=demo_scaler,
    kldiv_fn=kldiv_fn,
    egonce_fn=egonce_fn,
    alpha=0.05,
    device='cpu'
)
print(f'Average epoch loss: {avg_loss:.4f}')

---
## Section 7 — Evaluation & Metrics

### 7.1 Adaptive F1 Score

No fixed threshold is assumed. Instead:
1. Sweep a range of thresholds (e.g. 0.01–0.07 for Ego4D forecast).
2. For each threshold, binarize both predicted heatmap and GT heatmap.
3. Compute F1 = 2·P·R / (P+R).
4. Report the threshold that maximizes F1.

Only **fixation frames** (`gaze_type == 0`) are evaluated (saccades are excluded).

**Reference**: `slowfast/utils/metrics.py` lines 8–73

In [ ]:
# ── 7.2  Adaptive F1 implementation ────────────────────────────────────────────
# Reference: slowfast/utils/metrics.py  adaptive_f1()  (lines 8-73)

def adaptive_f1(preds, labels_hm, labels, dataset='ego4d_forecast'):
    """
    Compute the F1 score at the best probability threshold.

    Args:
        preds     : (B, 1, T, H, W) — probability maps (after softmax)
        labels_hm : (B,    T, H, W) — GT Gaussian heatmaps
        labels    : (B,    T, 3)    — [x, y, gaze_type], type=0 means fixation
        dataset   : controls threshold sweep range
    Returns:
        best_f1, best_recall, best_precision, best_threshold
    """
    # Threshold ranges (from slowfast/utils/metrics.py)
    thresh_ranges = {
        'ego4d_forecast': np.linspace(0.01, 0.07, 31),
        'aria_forecast' : np.linspace(0.00, 0.02, 21),
        'estimation'    : np.linspace(0.00, 0.02, 11),
    }
    thresholds = thresh_ranges.get(dataset, np.linspace(0.01, 0.07, 31))

    B, _, T, H, W = preds.shape
    preds_np   = preds.squeeze(1).detach().cpu().numpy()  # (B, T, H, W)
    labels_np  = labels_hm.detach().cpu().numpy()         # (B, T, H, W)
    gtype_np   = labels[:, :, 2].detach().cpu().numpy()   # (B, T) gaze type

    best_f1 = best_rec = best_prec = best_thr = 0.0

    for thr in thresholds:
        tp = fp = fn = 0
        for b in range(B):
            for t in range(T):
                # Skip non-fixation frames
                if int(gtype_np[b, t]) != 0:
                    continue
                pred_bin  = (preds_np[b, t]  >= thr).astype(np.float32)
                label_bin = (labels_np[b, t] >= thr).astype(np.float32)
                tp += (pred_bin * label_bin).sum()
                fp += (pred_bin * (1 - label_bin)).sum()
                fn += ((1 - pred_bin) * label_bin).sum()

        eps = 1e-8
        recall    = tp / (tp + fn + eps)
        precision = tp / (tp + fp + eps)
        f1        = 2 * recall * precision / (recall + precision + eps)
        if f1 > best_f1:
            best_f1, best_rec, best_prec, best_thr = f1, recall, precision, thr

    return best_f1, best_rec, best_prec, best_thr


# ── Demo ──────────────────────────────────────────────────────────────────────
preds_prob = torch.rand(B, 1, T, 64, 64)
preds_prob = preds_prob / (preds_prob.sum(dim=(-2,-1), keepdim=True) + 1e-8)
labels_hm_test = torch.rand(B, T, 64, 64)
labels_hm_test /= (labels_hm_test.sum(dim=(-2,-1), keepdim=True) + 1e-8)
labels_test = torch.zeros(B, T, 3)  # all fixations (gaze_type=0)

f1, rec, prec, thr = adaptive_f1(preds_prob, labels_hm_test, labels_test, 'ego4d_forecast')
print(f'Adaptive F1 : {f1:.4f}')
print(f'Recall      : {rec:.4f}')
print(f'Precision   : {prec:.4f}')
print(f'Best thresh : {thr:.4f}')

---
## Section 8 — Running Inference

### 8.1 Inference pipeline overview

```
1. Load video clip → decode 8 frames at target FPS
2. Load audio → compute STFT → crop to matching window
3. Preprocess: center crop 256×256, normalize pixels
4. model.eval(); torch.no_grad()
5. preds = model(frames, audio)
6. frame_softmax(preds, temperature=2)
7. Visualize per-frame heatmap
```

**Reference**: `tools/test_avgaze_net.py` perform_test() lines 21–93

In [ ]:
# ── 8.2  Inference skeleton ────────────────────────────────────────────────────

@torch.no_grad()
def run_inference(model, frames, audio, temperature=2.0, device='cpu'):
    """Run gaze forecasting inference.

    Args:
        model   : trained CSTS model
        frames  : (1, 3, T, H, W)  — single clip, preprocessed
        audio   : (1, 1, F, L)     — matching audio spectrogram
        temperature : softmax temperature
    Returns:
        heatmaps : (1, T, H/4, W/4) — predicted gaze probability maps
    """
    model.eval()
    frames = frames.to(device)
    audio  = audio.to(device)

    # Real call: preds = model([frames], audio)   (model expects list input)
    # Here we simulate the output shape:
    preds = torch.randn(1, 1, T, 64, 64, device=device)  # (B,1,T,H/4,W/4)

    # Per-frame softmax → probability map
    heatmaps = frame_softmax(preds, temperature)
    return heatmaps.squeeze(1)  # (1, T, 64, 64)


# ── 8.3  Visualize predicted heatmaps ─────────────────────────────────────────
import matplotlib.pyplot as plt

dummy_frames = torch.randn(1, 3, T, H, W)
dummy_audio  = torch.randn(1, 1, F_bins, L_time)
dummy_model  = nn.Identity()   # placeholder

heatmaps = run_inference(dummy_model, dummy_frames, dummy_audio)
print('Heatmap output:', heatmaps.shape)  # (1, 8, 64, 64)

fig, axes = plt.subplots(1, T, figsize=(16, 2))
for t in range(T):
    axes[t].imshow(heatmaps[0, t].numpy(), cmap='hot', vmin=0)
    axes[t].set_title(f't={t}')
    axes[t].axis('off')
plt.suptitle('Predicted Gaze Heatmaps (per frame)', y=1.02)
plt.tight_layout()
plt.savefig('gaze_heatmaps.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 9 — Configuration System

CSTS uses a YAML config system built on [fvcore](https://github.com/facebookresearch/fvcore).

### 9.1 Full annotated config (Ego4D Gaze Forecast)

**Reference**: `configs/Ego4D/CSTS_Ego4D_Gaze_Forecast.yaml`

In [ ]:
# ── 9.2  Annotated config (as Python dict for readability) ─────────────────────

config = {
    # ── Dataset ──────────────────────────────────────────────────────────────
    'TRAIN': {
        'ENABLE'            : True,
        'DATASET'           : 'ego4d_av_gaze_forecast',  # dataset registry key
        'BATCH_SIZE'        : 8,     # per-GPU batch size
        'EVAL_PERIOD'       : 1,     # run eval every N epochs
        'CHECKPOINT_PERIOD' : 1,     # save checkpoint every N epochs
        'AUTO_RESUME'       : True,  # resume from latest checkpoint
    },
    # ── Input Data ────────────────────────────────────────────────────────────
    'DATA': {
        'NUM_FRAMES'         : 8,       # T  — temporal frames per clip
        'SAMPLING_RATE'      : 8,       # stride between sampled frames (Ego4D @30fps)
        'TRAIN_CROP_SIZE'    : 256,     # H=W for training
        'TEST_CROP_SIZE'     : 256,     # H=W for testing
        'TARGET_FPS'         : 30,      # video FPS to normalize to
        'GAUSSIAN_KERNEL'    : 19,      # size of Gaussian heatmap kernel
        'PATH_PREFIX'        : '/path/to/Ego4D/clips.gaze',  # dataset root
    },
    # ── Model Architecture ────────────────────────────────────────────────────
    'MODEL': {
        'ARCH'               : 'mvit',
        'MODEL_NAME'         : 'CSTS',
        'LOSS_FUNC'          : 'kldiv+egonce',  # or 'kldiv'
        'LOSS_ALPHA'         : 0.05,    # weight on EgoNCE contrastive loss
        'DROPOUT_RATE'       : 0.5,
    },
    # ── MViT (Vision Transformer) ─────────────────────────────────────────────
    'MVIT': {
        'DEPTH'              : 16,       # number of transformer blocks
        'NUM_HEADS'          : 1,        # base number of heads (expands via HEAD_MUL)
        'EMBED_DIM'          : 96,       # base embedding dimension
        'PATCH_KERNEL'       : (3,7,7),  # 3-D patch embedding kernel
        'PATCH_STRIDE'       : (2,4,4),  # 3-D patch embedding stride
        'PATCH_PADDING'      : (1,3,3),  # 3-D patch embedding padding
        'MLP_RATIO'          : 4.0,      # FFN hidden dim = embed_dim * mlp_ratio
        'DROPPATH_RATE'      : 0.2,      # stochastic depth drop rate
        'DIM_MUL'            : [[1,2.0],[3,2.0],[14,2.0]],  # dim doubles at block 1,3,14
        'HEAD_MUL'           : [[1,2.0],[3,2.0],[14,2.0]],  # heads double at block 1,3,14
        'POOL_KV_STRIDE_ADAPTIVE': [1,8,8],  # K/V pooling stride (adaptive)
        'SEP_POS_EMBED'      : True,     # separate spatial+temporal positional embeddings
    },
    # ── Optimizer ────────────────────────────────────────────────────────────
    'SOLVER': {
        'BASE_LR'            : 1e-4,
        'LR_POLICY'          : 'cosine',
        'COSINE_END_LR'      : 1e-6,
        'WARMUP_EPOCHS'      : 0.0,
        'MAX_EPOCH'          : 15,
        'WEIGHT_DECAY'       : 0.05,
        'OPTIMIZING_METHOD'  : 'adamw',
        'CLIP_GRAD_L2NORM'   : 1.0,     # gradient clipping (L2 norm)
        'ZERO_WD_1D_PARAM'   : True,    # no WD for 1-D params (bias/norm)
        'ZERO_DECAY_POS_CLS' : True,    # no WD for pos-embed
    },
    # ── Test / Evaluation ─────────────────────────────────────────────────────
    'TEST': {
        'BATCH_SIZE'         : 96,
        'NUM_SPATIAL_CROPS'  : 1,
        'NUM_ENSEMBLE_VIEWS' : 1,
    },
    # ── Distributed Training ──────────────────────────────────────────────────
    'NUM_GPUS' : 4,   # total GPUs across nodes
    'RNG_SEED' : 42,
}

print('Key config values:')
for section in ['DATA','MVIT','SOLVER','MODEL']:
    print(f'\n[{section}]')
    for k, v in config[section].items():
        print(f'  {k:30s}: {v}')

---
## Section 10 — Putting It All Together

### Complete Shape Flow (End-to-End)

```
INPUTS
  frames       : (B=8, C=3,  T=8,  H=256, W=256)      RGB video
  audio        : (B=8, C=1,  F=128,       L=256)       STFT spectrogram

PATCH EMBEDDING  (PatchEmbed: kernel=(3,7,7), stride=(2,4,4), padding=(1,3,3))
  vis_tokens   : (B, 4×64×64=16384, 96)                [after Conv3d + LayerNorm]
  aud_tokens   : (B, 4×64×64=16384, 96)                [audio upsampled to 256×256]

POSITIONAL EMBEDDING  (separate spatial + temporal)
  → shapes unchanged, adds: pos_spatial(1,4096,96) + pos_temporal(1,4,96)

VISION ENCODER  (16 MultiScaleBlocks, Q-pooling reduces tokens)
  After block 1  : (B, 16384, 96)   dim×1, head×1
  After block 3  : (B,  4096, 192)  dim×2, head×2  (Q-pool [1,1,2,2])
  After block 14 : (B,  1024, 384)  dim×4, head×4
  After block 16 : (B,   256, 768)  dim×8, head×8

AUDIO ENCODER  (4 MultiScaleBlocks)
  Block 0 out  : (B, 16384, 192)
  Block 1 out  : (B,  4096, 384)
  Block 2 out  : (B,  1024, 768)
  Block 3 out  : (B,   256, 768)

SPATIAL FUSION
  audio pooled       : (B, 4, 768)       [Conv3d pool (1,8,8)]
  concat with vision : (B, 260, 768)
  after SpatialBlock : (B, 260, 768)
  vision output      : (B, 256, 768)

TEMPORAL FUSION
  vis temporal pool  : (B, 4, 768)
  aud temporal pool  : (B, 4, 768)
  concat             : (B, 8,  768)
  after TemporalBlock: (B, 8,  768)      [splits back → weights for reweighting]
  vis_reweighted     : (B, 256, 768)     [vision tokens × temporal attention weights]

DECODER  (4 MultiScaleBlocks, Q-upsampling expands tokens)
  Dec 1 out : (B,  1024, 768)  [Q-stride [1,2,2] → 4× more tokens]
  Dec 2 out : (B,  4096, 384)
  Dec 3 out : (B, 16384, 192)
  Dec 4 out : (B, 32768,  96)  [Q-stride [2,1,1] → temporal upsample]

RESHAPE  32768 = 8 × 64 × 64
  → (B, 96, 8, 64, 64)

CLASSIFIER  Conv3d(96→1, kernel=1)
  preds_logits : (B, 1, 8, 64, 64)

SOFTMAX  (per-frame, temperature=2)
  preds_prob   : (B, 1, 8, 64, 64)   ← final heatmap

LOSS COMPUTATION
  KLDiv  (preds_prob, labels_hm)    labels_hm : (B, 8, 64, 64)
  EgoNCE (v_embed @ a_embed.T)      embeddings: (B, D)
  total  = L_kl + 0.05 × L_nce
```

In [ ]:
# ── 10.1  Command-line training & evaluation (reference only) ─────────────────

TRAIN_CMD = """
# Train CSTS on Ego4D with 2 GPUs
CUDA_VISIBLE_DEVICES=0,1 python tools/run_net.py \\
    --init_method tcp://localhost:9880 \\
    --cfg configs/Ego4D/CSTS_Ego4D_Gaze_Forecast.yaml \\
    TRAIN.BATCH_SIZE 8 \\
    TEST.ENABLE False \\
    NUM_GPUS 2 \\
    DATA.PATH_PREFIX /path/to/Ego4D/clips.gaze \\
    TRAIN.CHECKPOINT_FILE_PATH /path/to/K400_MVIT_B_16x4_CONV.pyth \\
    OUTPUT_DIR out/csts_ego4d \\
    MODEL.LOSS_FUNC kldiv+egonce \\
    MODEL.LOSS_ALPHA 0.05 \\
    RNG_SEED 42
"""

TEST_CMD = """
# Evaluate CSTS on Ego4D with 1 GPU
CUDA_VISIBLE_DEVICES=0 python tools/run_net.py \\
    --cfg configs/Ego4D/CSTS_Ego4D_Gaze_Forecast.yaml \\
    TRAIN.ENABLE False \\
    TEST.BATCH_SIZE 24 \\
    NUM_GPUS 1 \\
    DATA.PATH_PREFIX /path/to/Ego4D/clips.gaze \\
    TEST.CHECKPOINT_FILE_PATH out/csts_ego4d/checkpoints/checkpoint_epoch_00015.pyth \\
    OUTPUT_DIR out/csts_ego4d/test
"""

print('TRAINING COMMAND')
print('=' * 60)
print(TRAIN_CMD)
print('EVALUATION COMMAND')
print('=' * 60)
print(TEST_CMD)

In [ ]:
# ── 10.2  Full end-to-end summary diagram ─────────────────────────────────────

summary = """
╔══════════════════════════════════════════════════════════════════╗
║              CSTS  — Complete Architecture Summary               ║
╠══════════════════════════════════════════════════════════════════╣
║  TASK    : Egocentric gaze forecasting (predict where to look)   ║
║  INPUTS  : RGB video (8 frames @ 30fps) + audio STFT spectrogram ║
║  OUTPUT  : Probability heatmap per frame (64×64)                 ║
╠══════════════════════════════════════════════════════════════════╣
║  COMPONENT            SHAPE               PARAMS                 ║
║  ─────────────────────────────────────────────────────────────── ║
║  Input frames         (B, 3, 8, 256, 256)                        ║
║  Input audio          (B, 1, 128, 256)                           ║
║  Patch tokens         (B, 16384, 96)      kernel=(3,7,7)         ║
║  Vision enc out       (B, 256, 768)       16 MViT blocks         ║
║  Audio enc out        (B, 256, 768)       4  MViT blocks         ║
║  Fused (spatial)      (B, 260, 768)       concat + attn          ║
║  Fused (temporal)     (B, 8,   768)       concat + attn          ║
║  Decoder out          (B, 32768, 96)      4 upsampling blocks    ║
║  Logits               (B, 1, 8, 64, 64)  Conv3d(96→1)           ║
╠══════════════════════════════════════════════════════════════════╣
║  LOSS FUNCTION                                                   ║
║    Primary  : KL Divergence  (pred heatmap vs GT Gaussian)       ║
║    Auxiliary: EgoNCE         (audio-visual contrastive, α=0.05)  ║
╠══════════════════════════════════════════════════════════════════╣
║  OPTIMIZER                                                       ║
║    AdamW (lr=1e-4, wd=0.05) + cosine decay to 1e-6, 15 epochs   ║
║    Gradient clipping: L2 norm ≤ 1.0                              ║
╠══════════════════════════════════════════════════════════════════╣
║  EVALUATION METRIC                                               ║
║    Adaptive F1 (best threshold from range sweep)                 ║
║    Fixation-only evaluation (gaze_type == 0)                     ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(summary)

---
## Section 11 — Exercises

Try these to deepen your understanding:

1. **Change the temperature** in `frame_softmax` from 2 to 1 or 4. How does it affect the sharpness of heatmaps?

2. **Ablate the EgoNCE loss**: set `alpha=0` and retrain. Does F1 drop on the Ego4D dataset?

3. **Vary the Gaussian kernel size** (7 vs 19 vs 31). How does it affect KLDiv loss values during early training?

4. **Single-modality baselines**: remove the audio encoder (set `audio_frames = torch.zeros_like(audio_frames)`) and observe the prediction quality change.

5. **Temporal fusion ablation**: skip the reweighting step and feed `vis_out_sp` directly to the decoder. Does performance change?

6. **Reproduce the config from scratch**: create `my_config.yaml` and override one key (e.g. `SOLVER.BASE_LR 5e-4`) via the CLI and verify the change is picked up.

---
## Key File Reference

| Component | File | Lines |
|-----------|------|-------|
| Entry point | `tools/run_net.py` | 11–29 |
| Train loop | `tools/train_avgaze_net.py` | 25–361 |
| Eval loop | `tools/test_avgaze_net.py` | 21–93 |
| CSTS model | `slowfast/models/custom_multimodal_builder.py` | 20–499 |
| Losses | `slowfast/models/losses.py` | 51–82, 152–170 |
| Optimizer | `slowfast/models/optimizer.py` | 11–108 |
| LR policy | `slowfast/utils/lr_policy.py` | 29–52 |
| Ego4D dataset | `slowfast/datasets/ego4d_avgaze_forecast.py` | 125–337 |
| Aria dataset | `slowfast/datasets/aria_avgaze_forecast.py` | 125–337 |
| Metrics | `slowfast/utils/metrics.py` | 8–73 |
| Default config | `slowfast/config/defaults.py` | all |
| Ego4D config | `configs/Ego4D/CSTS_Ego4D_Gaze_Forecast.yaml` | all |
| Aria config | `configs/Aria/CSTS_Aria_Gaze_Forecast.yaml` | all |